## ML_1014: Implement LayerNorm

**Difficulty**: Medium | **TorchCode**: #04

### Problem
Implement Layer Normalization from scratch. Normalize over the last dimension, then apply learnable scale (γ) and shift (β).

### Formula
$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \varepsilon}} + \beta, \quad \mu = \frac{1}{d}\sum x_i, \quad \sigma^2 = \frac{1}{d}\sum (x_i - \mu)^2$$

In [ ]:
import torch

def my_layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(dim=-1, keepdim=True)
    # both equal: var = x.var(dim=-1, keepdim=True, unbiased=False)
    var = ((x - mean) ** 2).mean(dim=-1, keepdim=True)
    x_norm = (x - mean) / torch.sqrt(var + eps)
    return gamma * x_norm + beta

In [ ]:
import torch

# --- Test 1: Shape and correctness vs F.layer_norm ---
x = torch.randn(2, 3, 8)
gamma = torch.ones(8)
beta = torch.zeros(8)
out = my_layer_norm(x, gamma, beta)
assert out.shape == x.shape
ref = torch.nn.functional.layer_norm(x, [8], gamma, beta)
assert torch.allclose(out, ref, atol=1e-4)

# --- Test 2: With learned parameters ---
x = torch.randn(4, 16)
gamma = torch.randn(16)
beta = torch.randn(16)
out = my_layer_norm(x, gamma, beta)
ref = torch.nn.functional.layer_norm(x, [16], gamma, beta)
assert torch.allclose(out, ref, atol=1e-4)

# --- Test 3: Gradient flow ---
x = torch.randn(2, 8, requires_grad=True)
gamma = torch.ones(8, requires_grad=True)
beta = torch.zeros(8, requires_grad=True)
my_layer_norm(x, gamma, beta).sum().backward()
assert x.grad is not None
assert gamma.grad is not None

print('All tests passed!')